In [1]:
# ============================================================
# MD6 — DISTILROBERTA + SOFTENED CLASS-WEIGHTED CE
#
# Research question:
#   Can a softer class-weighting strength keep MD4's gains on
#   probing/telling while recovering focus performance and accuracy?
#
# ONLY MAJOR EXPERIMENTAL CHANGE FROM REPRODUCED MD4:
#   full inverse-frequency weights (alpha=1.0)
#       ->
#   softened weights: raw_weight ** 0.75
#
# Everything else is held fixed:
#   Model: distilbert/distilroberta-base
#   Max length: 512
#   LR: 2e-5
#   Effective global batch: 16 (2 GPUs x batch 8)
#   Epochs: 3
#   Weight decay: 0.01
#   Warmup steps: 280
#   Seed: 42
#
# Primary metric: Macro-F1
# Secondary metric: Accuracy
# Test set: NOT USED
# ============================================================

import gc
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LENGTH = 512

# TUNED PARAMETER FOR THIS NOTEBOOK
WEIGHT_ALPHA = 0.75

TRAIN_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl"
)
VAL_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl"
)

OUTPUT_DIR = Path("/kaggle/working/md6_soft_weights_alpha075")
RESULT_DIR = Path("/kaggle/working/md6_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["generic", "probing", "focus", "telling"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 78)
print("MD6 CONFIGURATION")
print("=" * 78)
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Weight alpha:", WEIGHT_ALPHA)
print("GPU count:", torch.cuda.device_count())

# Keep the hardware setup consistent with the reproduced MD4 experiment.
assert torch.cuda.is_available(), "GPU is required."
assert torch.cuda.device_count() == 2, (
    "This experiment is defined for Kaggle 2xT4. "
    f"Detected {torch.cuda.device_count()} GPU(s)."
)

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

assert len(train_rows) == 14905
assert len(val_rows) == 1850

required = {"example_id", "qid", "problem", "history", "target_move"}
for name, rows in [("train", train_rows), ("validation", val_rows)]:
    assert required.issubset(rows[0].keys()), f"{name} schema mismatch"
    bad = [r for r in rows if r["target_move"] not in LABEL2ID]
    assert not bad, f"{name}: invalid labels found"

print("Train examples:", len(train_rows))
print("Validation examples:", len(val_rows))
print("Test set used: False")


MD6 CONFIGURATION
Model: distilbert/distilroberta-base
Max length: 512
Weight alpha: 0.75
GPU count: 2
Train examples: 14905
Validation examples: 1850
Test set used: False


In [2]:
# ============================================================
# INPUT FORMAT + TOKENIZATION + SOFTENED CLASS WEIGHTS
# ============================================================

def format_history(history):
    if not history:
        return "[No previous conversation]"
    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )

def build_problem_text(example):
    return "Problem:\n" + str(example["problem"])

def build_conversation_text(example):
    return (
        "Conversation:\n"
        + format_history(example["history"])
        + "\n\nNext teacher pedagogical move:"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

def prepare_rows(rows):
    prepared = []
    for ex in rows:
        prepared.append({
            "example_id": ex["example_id"],
            "qid": ex["qid"],
            "problem_text": build_problem_text(ex),
            "conversation_text": build_conversation_text(ex),
            "labels": LABEL2ID[ex["target_move"]],
        })
    return prepared

train_prepared = prepare_rows(train_rows)
val_prepared = prepare_rows(val_rows)

# Fixed DistilRoBERTa short/long diagnostic split.
val_original_lengths = []
for ex in val_rows:
    encoded = tokenizer(
        build_problem_text(ex),
        build_conversation_text(ex),
        add_special_tokens=True,
        truncation=False,
    )
    val_original_lengths.append(len(encoded["input_ids"]))

val_original_lengths = np.asarray(val_original_lengths)
short_mask = val_original_lengths <= 512
long_mask = val_original_lengths > 512

print("Validation <=512:", int(short_mask.sum()))
print("Validation >512:", int(long_mask.sum()))
assert int(short_mask.sum()) == 1507
assert int(long_mask.sum()) == 343

# ------------------------------------------------------------
# Raw inverse-frequency weights = reproduced MD4 weighting.
# MD6 softens the RELATIVE weighting with exponent alpha=.75.
#
# alpha=0.0 -> ordinary CE (all weights become 1)
# alpha=1.0 -> reproduced MD4 full inverse-frequency weighting
# alpha=0.5 -> three-quarter-strength experiment used here
# ------------------------------------------------------------

train_counts = Counter(r["target_move"] for r in train_rows)
n = len(train_rows)
k = len(LABELS)

raw_class_weights = np.asarray(
    [n / (k * train_counts[label]) for label in LABELS],
    dtype=np.float32,
)

class_weights = np.power(raw_class_weights, WEIGHT_ALPHA).astype(np.float32)

print("\n" + "=" * 78)
print("CLASS WEIGHTS")
print("=" * 78)
for label, raw_w, tuned_w in zip(LABELS, raw_class_weights, class_weights):
    print(
        f"{label:10s} count={train_counts[label]:4d} "
        f"raw={raw_w:.6f} tuned={tuned_w:.6f}"
    )

train_ds = Dataset.from_list(train_prepared)
val_ds = Dataset.from_list(val_prepared)

def tokenize_batch(batch):
    return tokenizer(
        batch["problem_text"],
        batch["conversation_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["problem_text", "conversation_text", "example_id", "qid"],
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["problem_text", "conversation_text", "example_id", "qid"],
)

assert max(len(x) for x in train_tok["input_ids"]) <= MAX_LENGTH
assert max(len(x) for x in val_tok["input_ids"]) <= MAX_LENGTH

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nTokenization: PASSED")


Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


Validation <=512: 1507
Validation >512: 343

CLASS WEIGHTS
generic    count=3579 raw=1.041143 tuned=1.030701
probing    count=3413 raw=1.091781 tuned=1.068075
focus      count=5443 raw=0.684595 tuned=0.752619
telling    count=2470 raw=1.508603 tuned=1.361229


Map:   0%|          | 0/14905 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]


Tokenization: PASSED


In [3]:
# ============================================================
# MODEL + WEIGHTED TRAINER + METRICS
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = torch.tensor(
            class_weights, dtype=torch.float32
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights_tensor.to(logits.device)

        loss = F.cross_entropy(
            logits,
            labels,
            weight=weights,
        )

        if return_outputs:
            return loss, outputs
        return loss

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    pred = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(
            labels,
            pred,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        ),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # EXACTLY PRESERVED FROM REPRODUCED MD4
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=280,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    # Select the best checkpoint using the PRIMARY research metric.
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print("Trainer GPU count:", trainer.args.n_gpu)
assert trainer.args.n_gpu == 2


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer GPU count: 2


In [4]:
# ============================================================
# TRAIN
# ============================================================

print("\n" + "=" * 78)
print("STARTING MD6 TRAINING")
print("=" * 78)

train_result = trainer.train()

# Preserve full Trainer history.
history_path = RESULT_DIR / "md6_log_history.json"
with history_path.open("w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print("\n" + "=" * 78)
print("EPOCH VALIDATION HISTORY")
print("=" * 78)

for row in trainer.state.log_history:
    if "eval_macro_f1" in row:
        print(
            f"Epoch {row.get('epoch')}: "
            f"accuracy={row.get('eval_accuracy'):.6f} "
            f"macro_f1={row.get('eval_macro_f1'):.6f} "
            f"val_loss={row.get('eval_loss'):.6f}"
        )

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("Best Macro-F1:", trainer.state.best_metric)



STARTING MD6 TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.231116,1.211870,0.445405,0.444903
2,1.153606,1.186119,0.501622,0.487685
3,1.093237,1.201221,0.501622,0.484501


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


EPOCH VALIDATION HISTORY
Epoch 1.0: accuracy=0.445405 macro_f1=0.444903 val_loss=1.211870
Epoch 2.0: accuracy=0.501622 macro_f1=0.487685 val_loss=1.186119
Epoch 3.0: accuracy=0.501622 macro_f1=0.484501 val_loss=1.201221

Best checkpoint: /kaggle/working/md6_soft_weights_alpha075/checkpoint-1864
Best Macro-F1: 0.487684572655523


In [5]:
# =========================================================
# EVALUATE ALL SAVED EPOCH CHECKPOINTS + SAVE AUDIT ARTIFACTS
# ==========================================================

def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

def subset_metrics(true_ids, pred_ids, mask):
    y_true = true_ids[mask]
    y_pred = pred_ids[mask]
    return {
        "n": int(mask.sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(range(len(LABELS))),
                average="macro",
                zero_division=0,
            )
        ),
    }

checkpoint_dirs = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)

assert len(checkpoint_dirs) == 3, (
    f"Expected 3 checkpoints, found {len(checkpoint_dirs)}"
)

all_results = []

for epoch_idx, ckpt in enumerate(checkpoint_dirs, start=1):
    print("\n" + "-" * 78)
    print(f"EPOCH {epoch_idx}: {ckpt}")
    print("-" * 78)

    ckpt_model = AutoModelForSequenceClassification.from_pretrained(ckpt)

    eval_args = TrainingArguments(
        output_dir=str(RESULT_DIR / f"eval_epoch{epoch_idx}"),
        per_device_eval_batch_size=16,
        fp16=True,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )

    eval_trainer = WeightedTrainer(
        model=ckpt_model,
        args=eval_args,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    pred_output = eval_trainer.predict(val_tok)
    logits = pred_output.predictions
    true_ids = pred_output.label_ids
    pred_ids = np.argmax(logits, axis=1)
    probs = softmax_np(logits)

    overall_accuracy = float(accuracy_score(true_ids, pred_ids))
    overall_macro_f1 = float(
        f1_score(
            true_ids,
            pred_ids,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        )
    )

    per_class_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
        average=None,
        zero_division=0,
    )

    report = classification_report(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
        target_names=LABELS,
        digits=6,
        zero_division=0,
        output_dict=True,
    )

    cm = confusion_matrix(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
    ).tolist()

    short_result = subset_metrics(true_ids, pred_ids, short_mask)
    long_result = subset_metrics(true_ids, pred_ids, long_mask)

    result = {
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),
        "accuracy": overall_accuracy,
        "macro_f1": overall_macro_f1,
        "per_class_f1": {
            label: float(score)
            for label, score in zip(LABELS, per_class_f1)
        },
        "classification_report": report,
        "confusion_matrix": cm,
        "le512": short_result,
        "gt512": long_result,
    }
    all_results.append(result)

    print("Accuracy:", overall_accuracy)
    print("Macro-F1:", overall_macro_f1)
    print("<=512 Macro-F1:", short_result["macro_f1"])
    print(">512 Macro-F1:", long_result["macro_f1"])
    print("Per-class F1:", result["per_class_f1"])

    pred_path = RESULT_DIR / f"md6_epoch{epoch_idx}_predictions.jsonl"
    with pred_path.open("w", encoding="utf-8") as f:
        for i, (true_id, pred_id, probability_row) in enumerate(
            zip(true_ids, pred_ids, probs)
        ):
            record = {
                "example_id": val_rows[i]["example_id"],
                "qid": val_rows[i]["qid"],
                "true_move": ID2LABEL[int(true_id)],
                "predicted_move": ID2LABEL[int(pred_id)],
                "original_distil_token_length": int(val_original_lengths[i]),
                "probabilities": {
                    label: float(probability_row[j])
                    for j, label in enumerate(LABELS)
                },
            }
            f.write(json.dumps(record) + "\n")

    del eval_trainer, ckpt_model
    gc.collect()
    torch.cuda.empty_cache()

best_result = max(all_results, key=lambda x: x["macro_f1"])

summary = {
    "experiment": "MD6_softened_inverse_frequency_weights",
    "research_question": (
        "Can alpha=0.75 softened class weighting improve the "
        "Macro-F1/accuracy trade-off relative to full MD4 weighting?"
    ),
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "weight_alpha": WEIGHT_ALPHA,
    "raw_class_weights": {
        label: float(w)
        for label, w in zip(LABELS, raw_class_weights)
    },
    "tuned_class_weights": {
        label: float(w)
        for label, w in zip(LABELS, class_weights)
    },
    "training_configuration": {
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 8,
        "effective_global_batch": 16,
        "num_train_epochs": 3,
        "weight_decay": 0.01,
        "warmup_steps": 280,
        "fp16": True,
        "seed": SEED,
        "test_used": False,
    },
    "baseline_reproduced_md4_epoch3": {
        "accuracy": 0.4897297297297297,
        "macro_f1": 0.48321092131167,
    },
    "epochs": all_results,
    "best_epoch": best_result["epoch"],
    "best_accuracy": best_result["accuracy"],
    "best_macro_f1": best_result["macro_f1"],
}

summary_path = RESULT_DIR / "md6_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Save a clean deployable copy of the winning checkpoint.
best_checkpoint = Path(best_result["checkpoint"])
best_model = AutoModelForSequenceClassification.from_pretrained(best_checkpoint)

best_model_dir = RESULT_DIR / "md6_best_model"
best_model.save_pretrained(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

archive_path = shutil.make_archive(
    str(RESULT_DIR / "md6_best_model"),
    "gztar",
    root_dir=best_model_dir,
)

print("\n" + "=" * 78)
print("MD6 COMPLETE")
print("=" * 78)
print("Best epoch:", best_result["epoch"])
print("Best accuracy:", best_result["accuracy"])
print("Best Macro-F1:", best_result["macro_f1"])
print("Summary:", summary_path)
print("Best model archive:", archive_path)

print("\nDecision reference:")
print("Reproduced MD4 Epoch 3 accuracy = 0.4897297297297297")
print("Reproduced MD4 Epoch 3 Macro-F1 = 0.48321092131167")



------------------------------------------------------------------------------
EPOCH 1: /kaggle/working/md6_soft_weights_alpha075/checkpoint-932
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.4454054054054054
Macro-F1: 0.44490319209751006
<=512 Macro-F1: 0.46573180797666686
>512 Macro-F1: 0.13315093479252166
Per-class F1: {'generic': 0.5809906291834003, 'probing': 0.335832083958021, 'focus': 0.465, 'telling': 0.39779005524861877}

------------------------------------------------------------------------------
EPOCH 2: /kaggle/working/md6_soft_weights_alpha075/checkpoint-1864
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.5016216216216216
Macro-F1: 0.487684572655523
<=512 Macro-F1: 0.4903014915021776
>512 Macro-F1: 0.24157147800962345
Per-class F1: {'generic': 0.5875, 'probing': 0.4178712220762155, 'focus': 0.5710080941869021, 'telling': 0.37435897435897436}

------------------------------------------------------------------------------
EPOCH 3: /kaggle/working/md6_soft_weights_alpha075/checkpoint-2796
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.5016216216216216
Macro-F1: 0.48450137554189354
<=512 Macro-F1: 0.4867404501958344
>512 Macro-F1: 0.2509918018764886
Per-class F1: {'generic': 0.5945273631840796, 'probing': 0.4011299435028249, 'focus': 0.5688073394495413, 'telling': 0.3735408560311284}


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MD6 COMPLETE
Best epoch: 2
Best accuracy: 0.5016216216216216
Best Macro-F1: 0.487684572655523
Summary: /kaggle/working/md6_results/md6_summary.json
Best model archive: /kaggle/working/md6_results/md6_best_model.tar.gz

Decision reference:
Reproduced MD4 Epoch 3 accuracy = 0.4897297297297297
Reproduced MD4 Epoch 3 Macro-F1 = 0.48321092131167
